# Notebook 1: Extract and Load Raw Data
This notebook handles the extraction of raw data from a CSV file and loads it into the `raw_survey` table in a PostgreSQL database. It includes the following steps:
- Setting up a secure database connection using environment variables.
- Executing the `setup.py` script to create the database and table if they don’t exist.
- Loading the CSV data into a Pandas DataFrame.
- Checking if data already exists in the `raw_survey` table to avoid redundant loading.
- Loading the data into the database if necessary.
- Verifying the data load with a sample query.

## Step 1: Import Libraries and Set Up Database Connection
We import the required Python libraries and establish a connection to the PostgreSQL database using credentials stored in a `.env` file for security.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(DATABASE_URL)

## Step 2: Create Database and Table

We run the `setup.py` script to ensure the `dev_survey_insights` database and `raw_survey` table are created using SQLAlchemy.

In [ ]:
%run ../scripts/setup.py

## Step 3: Load CSV Data into DataFrame

The raw survey data is loaded from the CSV file into a Pandas DataFrame for processing.

In [ ]:
csv_path = '../data/survey_results_public.csv'

df = pd.read_csv(csv_path)
print(f"Data loaded successfully. Rows: {len(df)}, Columns: {len(df.columns)}")

## Step 4: Check for Existing Data in Database

To prevent redundant loading and potential crashes, we check if the `raw_survey` table already contains data. If it does, we skip the loading step.

In [ ]:
query = "SELECT COUNT(*) FROM raw_survey"
count = pd.read_sql(query, engine).iloc[0, 0]

if count > 0:
    print("Data already exists in the 'raw_survey' table. Skipping the load step.")
else:
    df.to_sql('raw_survey', engine, if_exists='replace', index=False)
    print("Data loaded into the 'raw_survey' table successfully.")

## Step 5: Verify Data Loading

We query the first 5 rows of the `raw_survey` table to confirm that the data was loaded correctly.

In [ ]:
query = "SELECT * FROM raw_survey LIMIT 5;"
pd.read_sql(query, engine)